# Разметка волновода (waveguide) в датасете

Как выглядят текущие Roboflow-аннотации класса **waveguide (class 0)** на кадрах из 4 видео.

In [ ]:
import os, cv2, glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly
from pathlib import Path

PROJECT = Path(os.path.abspath(''))
if not (PROJECT / 'config.py').exists():
    PROJECT = Path(os.path.expanduser('~/Desktop/НИР/ДаняБоряНир'))
DATA = PROJECT.parent / 'data' / 'dataset'
RESULTS = PROJECT / 'results'

IMG_DIR = DATA / 'images' / 'train'
LBL_DIR = DATA / 'labels' / 'train'

print(f'Images: {IMG_DIR}')
print(f'Labels: {LBL_DIR}')

In [ ]:
def parse_yolo_seg(lbl_path, class_filter=None):
    """Парсит YOLO-seg label файл. Возвращает [(class_id, [(x,y), ...]), ...]"""
    result = []
    if not lbl_path.exists():
        return result
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cls_id = int(parts[0])
            if class_filter is not None and cls_id != class_filter:
                continue
            coords = list(map(float, parts[1:]))
            pts = [(coords[i], coords[i+1]) for i in range(0, len(coords)-1, 2)]
            result.append((cls_id, pts))
    return result

def draw_annotation(ax, img_path, lbl_path, show_all=True):
    """Рисует кадр с аннотациями. Waveguide — красный, flux — голубой, solder — жёлтый."""
    img = cv2.imread(str(img_path))
    if img is None:
        ax.text(0.5, 0.5, 'Image not found', ha='center')
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    
    colors = {
        0: ('#FF4444', '#FF4444', 'waveguide'),
        1: ('#44DDDD', '#44DDDD', 'flux'),
        2: ('#FFDD33', '#FFDD33', 'solder'),
    }
    
    annotations = parse_yolo_seg(lbl_path)
    for cls_id, pts_norm in annotations:
        if not show_all and cls_id != 0:
            continue
        fc, ec, name = colors.get(cls_id, ('#888', '#888', f'cls{cls_id}'))
        # Денормализация
        pts_px = [(x * w, y * h) for x, y in pts_norm]
        
        if cls_id == 0:
            # Waveguide — жирный контур, полупрозрачная заливка
            poly = MplPoly(pts_px, closed=True,
                          facecolor=fc, edgecolor='red',
                          alpha=0.3, linewidth=2.5)
        else:
            # Остальные — тоньше и бледнее
            poly = MplPoly(pts_px, closed=True,
                          facecolor=fc, edgecolor=ec,
                          alpha=0.15, linewidth=1, linestyle='--')
        ax.add_patch(poly)
        
        # Подпись
        cx = np.mean([p[0] for p in pts_px])
        cy = np.min([p[1] for p in pts_px]) - 8
        fontsize = 10 if cls_id == 0 else 7
        ax.text(cx, cy, name, color=ec, fontsize=fontsize, ha='center',
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.2',
                facecolor='black', alpha=0.6))

## 1. Обзор по 4 видео — все 3 класса
Waveguide выделен красным (жирный контур), flux — голубой пунктир, solder — жёлтый пунктир.

In [ ]:
# По одному кадру из каждого видео — все классы
videos = ['MVI_6265', 'MVI_6268', 'MVI_6270', 'MVI_6273']

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
for ax, vid in zip(axes, videos):
    # Первый кадр с waveguide аннотацией
    found = False
    for lbl in sorted(LBL_DIR.glob(f'{vid}*.txt')):
        annots = parse_yolo_seg(lbl, class_filter=0)
        if annots:
            img_name = lbl.stem + '.jpg'
            img_path = IMG_DIR / img_name
            if img_path.exists():
                draw_annotation(ax, img_path, lbl, show_all=True)
                ax.set_title(f'{vid}\n({len(annots)} waveguide)', fontsize=11)
                found = True
                break
    if not found:
        ax.text(0.5, 0.5, f'{vid}\nне найден', ha='center', va='center')
    ax.axis('off')

plt.suptitle('Разметка по видео: waveguide (красный), flux (голубой), solder (жёлтый)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Только waveguide — крупный план
Показываем waveguide-полигон с зумом на область волновода.

In [ ]:
# 4 видео × 3 кадра (начало, середина, конец) — только waveguide
fig, axes = plt.subplots(4, 3, figsize=(18, 24))

for row, vid in enumerate(videos):
    # Собираем все label-файлы с waveguide для этого видео
    pairs = []
    for lbl in sorted(LBL_DIR.glob(f'{vid}*.txt')):
        annots = parse_yolo_seg(lbl, class_filter=0)
        if annots:
            img_path = IMG_DIR / (lbl.stem + '.jpg')
            if img_path.exists():
                pairs.append((img_path, lbl))
    
    # Берём начало, середину, конец
    if len(pairs) >= 3:
        indices = [0, len(pairs)//2, len(pairs)-1]
    elif len(pairs) == 2:
        indices = [0, 1, 1]
    elif len(pairs) == 1:
        indices = [0, 0, 0]
    else:
        indices = []
    
    for col, idx in enumerate(indices):
        ax = axes[row][col]
        img_path, lbl_path = pairs[idx]
        
        # Рисуем
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        ax.imshow(img)
        
        annots = parse_yolo_seg(lbl_path, class_filter=0)
        for cls_id, pts_norm in annots:
            pts_px = [(x * w, y * h) for x, y in pts_norm]
            poly = MplPoly(pts_px, closed=True,
                          facecolor='red', edgecolor='#FF0000',
                          alpha=0.3, linewidth=2)
            ax.add_patch(poly)
            
            # Зум на область waveguide
            xs = [p[0] for p in pts_px]
            ys = [p[1] for p in pts_px]
            margin = 60
            ax.set_xlim(min(xs) - margin, max(xs) + margin)
            ax.set_ylim(max(ys) + margin, min(ys) - margin)
            
            # Площадь полигона
            from matplotlib.path import Path as MplPath
            path = MplPath(pts_px)
            # Площадь через shoelace
            n_pts = len(pts_px)
            area = 0
            for i in range(n_pts):
                j = (i + 1) % n_pts
                area += pts_px[i][0] * pts_px[j][1]
                area -= pts_px[j][0] * pts_px[i][1]
            area_px = abs(area) / 2
            area_pct = area_px / (w * h) * 100
            
            ax.text(np.mean(xs), min(ys) - 12,
                    f'{len(pts_px)} точек, {area_pct:.1f}% кадра',
                    color='red', fontsize=9, ha='center', fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Номер кадра из имени файла
        frame_num = img_path.stem.split('-')[1][:4] if '-' in img_path.stem else '?'
        pos_label = ['начало', 'середина', 'конец'][col]
        ax.set_title(f'{vid} #{frame_num} ({pos_label})', fontsize=10)
        ax.axis('off')

plt.suptitle('Waveguide-полигоны (zoom): начало → середина → конец каждого видео', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Геометрия waveguide-полигона
Нормализованные координаты + bounding box + площадь.

In [ ]:
# Собираем статистику по всем waveguide аннотациям
wg_stats = []

for split in ['train', 'val', 'test']:
    lbl_dir = DATA / 'labels' / split
    if not lbl_dir.exists():
        continue
    for lbl in lbl_dir.glob('*.txt'):
        annots = parse_yolo_seg(lbl, class_filter=0)
        for cls_id, pts_norm in annots:
            xs = [p[0] for p in pts_norm]
            ys = [p[1] for p in pts_norm]
            # Shoelace для нормализованной площади
            n = len(pts_norm)
            area = 0
            for i in range(n):
                j = (i + 1) % n
                area += pts_norm[i][0] * pts_norm[j][1]
                area -= pts_norm[j][0] * pts_norm[i][1]
            area = abs(area) / 2
            
            vid = lbl.stem.split('_MOV')[0] if '_MOV' in lbl.stem else 'unk'
            wg_stats.append({
                'video': vid,
                'split': split,
                'n_points': len(pts_norm),
                'cx': np.mean(xs),
                'cy': np.mean(ys),
                'bbox_w': max(xs) - min(xs),
                'bbox_h': max(ys) - min(ys),
                'area_norm': area,
                'area_pct': area * 100,
            })

import pandas as pd
df = pd.DataFrame(wg_stats)
print(f'Всего waveguide аннотаций: {len(df)}')
print(f'\nСтатистика по видео:')
display(df.groupby('video').agg({
    'n_points': ['mean', 'min', 'max'],
    'area_pct': ['mean', 'min', 'max'],
    'bbox_w': 'mean',
    'bbox_h': 'mean',
    'cx': 'mean',
    'cy': 'mean',
}).round(3))

In [ ]:
# Визуализация распределений
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Площадь
axes[0,0].hist(df['area_pct'], bins=30, color='#FF6B6B', edgecolor='black', alpha=0.7)
axes[0,0].set_xlabel('Площадь waveguide (% кадра)')
axes[0,0].set_ylabel('Кадров')
axes[0,0].set_title('Распределение площади')
axes[0,0].axvline(df['area_pct'].mean(), color='red', ls='--', label=f'mean={df["area_pct"].mean():.1f}%')
axes[0,0].legend()

# Bbox ширина
axes[0,1].hist(df['bbox_w'], bins=30, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[0,1].set_xlabel('Ширина bbox (норм.)')
axes[0,1].set_title('Ширина bounding box')

# Bbox высота
axes[0,2].hist(df['bbox_h'], bins=30, color='#FFD93D', edgecolor='black', alpha=0.7)
axes[0,2].set_xlabel('Высота bbox (норм.)')
axes[0,2].set_title('Высота bounding box')

# Центр масс (scatter)
for vid, group in df.groupby('video'):
    axes[1,0].scatter(group['cx'], group['cy'], alpha=0.5, s=20, label=vid)
axes[1,0].set_xlabel('cx (норм.)')
axes[1,0].set_ylabel('cy (норм.)')
axes[1,0].set_title('Положение центра waveguide')
axes[1,0].set_xlim(0, 1)
axes[1,0].set_ylim(1, 0)  # инверсия y
axes[1,0].legend(fontsize=8)
axes[1,0].set_aspect('equal')

# Количество точек полигона
axes[1,1].hist(df['n_points'], bins=20, color='#9B59B6', edgecolor='black', alpha=0.7)
axes[1,1].set_xlabel('Количество точек полигона')
axes[1,1].set_title('Детализация контура waveguide')

# Площадь по видео (boxplot)
data_by_vid = [group['area_pct'].values for _, group in df.groupby('video')]
labels_vid = [vid for vid, _ in df.groupby('video')]
bp = axes[1,2].boxplot(data_by_vid, labels=labels_vid, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#FFD93D', '#9B59B6']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1,2].set_ylabel('Площадь (%)')
axes[1,2].set_title('Площадь waveguide по видео')

plt.suptitle('Статистика waveguide-аннотаций', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Наложение всех waveguide-полигонов на один кадр
Видно, как контур дрейфует между кадрами одного видео.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

for ax, vid in zip(axes, videos):
    # Берём первый кадр как фон
    bg_img = None
    for img_p in sorted(IMG_DIR.glob(f'{vid}*.jpg')):
        bg_img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
        break
    
    if bg_img is None:
        ax.text(0.5, 0.5, f'{vid} нет', ha='center')
        continue
    
    h, w = bg_img.shape[:2]
    ax.imshow(bg_img)
    
    # Все waveguide-полигоны для этого видео
    count = 0
    for lbl in sorted(LBL_DIR.glob(f'{vid}*.txt')):
        annots = parse_yolo_seg(lbl, class_filter=0)
        for _, pts_norm in annots:
            pts_px = [(x * w, y * h) for x, y in pts_norm]
            poly = MplPoly(pts_px, closed=True,
                          facecolor='none', edgecolor='red',
                          alpha=0.15, linewidth=0.8)
            ax.add_patch(poly)
            count += 1
    
    # Зум на waveguide-зону
    all_cx, all_cy = [], []
    for lbl in sorted(LBL_DIR.glob(f'{vid}*.txt')):
        annots = parse_yolo_seg(lbl, class_filter=0)
        for _, pts_norm in annots:
            all_cx.append(np.mean([p[0] for p in pts_norm]) * w)
            all_cy.append(np.mean([p[1] for p in pts_norm]) * h)
    
    if all_cx:
        mcx, mcy = np.mean(all_cx), np.mean(all_cy)
        span = 150
        ax.set_xlim(mcx - span, mcx + span)
        ax.set_ylim(mcy + span, mcy - span)
    
    ax.set_title(f'{vid}\n{count} waveguide полигонов', fontsize=11)
    ax.axis('off')

plt.suptitle('Все waveguide-контуры наложены (виден дрейф и разброс)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Сравнение: разметка Roboflow vs эталон
Слева — оригинальная аннотация на кадре, справа — ручная эталонная разметка.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Слева: пример Roboflow-аннотации
ax = axes[0]
for lbl in sorted(LBL_DIR.glob('MVI_6270*.txt')):
    annots = parse_yolo_seg(lbl, class_filter=0)
    if annots:
        img_path = IMG_DIR / (lbl.stem + '.jpg')
        if img_path.exists():
            draw_annotation(ax, img_path, lbl, show_all=True)
            ax.set_title('Roboflow-аннотация (оригинал)', fontsize=13)
            break
ax.axis('off')

# Справа: эталонная разметка
ax = axes[1]
ref_path = RESULTS / 'user_annotation_waveguide.jpg'
if ref_path.exists():
    img = plt.imread(str(ref_path))
    ax.imshow(img)
    ax.set_title('Эталон: ручная разметка waveguide', fontsize=13)
else:
    ax.text(0.5, 0.5, 'user_annotation_waveguide.jpg\nне найден', ha='center', va='center')
ax.axis('off')

plt.suptitle('Сравнение: Roboflow vs эталонная разметка', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Динамика waveguide по кадрам одного видео
Как меняется площадь и положение waveguide-полигона по ходу видео.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, vid in zip(axes.flat, videos):
    frames = []
    areas = []
    cxs = []
    cys = []
    
    for lbl in sorted(LBL_DIR.glob(f'{vid}*.txt')):
        annots = parse_yolo_seg(lbl, class_filter=0)
        if annots:
            _, pts = annots[0]
            xs = [p[0] for p in pts]
            ys = [p[1] for p in pts]
            n = len(pts)
            area = 0
            for i in range(n):
                j = (i + 1) % n
                area += pts[i][0] * pts[j][1]
                area -= pts[j][0] * pts[i][1]
            area = abs(area) / 2 * 100
            
            frame_str = lbl.stem.split('-')[1][:4] if '-' in lbl.stem else '0'
            try:
                frame_num = int(frame_str)
            except:
                frame_num = len(frames)
            
            frames.append(frame_num)
            areas.append(area)
            cxs.append(np.mean(xs))
            cys.append(np.mean(ys))
    
    if frames:
        ax.plot(frames, areas, 'o-', color='#FF4444', markersize=3, linewidth=1)
        ax.set_xlabel('Кадр')
        ax.set_ylabel('Площадь waveguide (%)')
        ax.set_title(f'{vid}: площадь waveguide по кадрам', fontsize=11)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{vid}: нет данных', ha='center', va='center')

plt.suptitle('Динамика площади waveguide-полигона по кадрам', fontsize=14)
plt.tight_layout()
plt.show()

## Итог

- **Waveguide** размечен полигонами из ~36-50 точек, занимает ~3-10% площади кадра
- Положение центра относительно стабильно (зависит от аугментации Roboflow)
- Между кадрами видна вариативность контуров — нужна проверка на корректность границ
- Эталонная разметка показывает, что waveguide — это **малая прямоугольная компонента** с чёткими границами металлических стенок